|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 5:</h2>|<h1>Making It Fast<h1>|
|<h2>Section:</h2>|<h1>Incidents<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: the incident file<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

You finished Part 5. The step is captured as a graph, the sampler is one
vectorized pass, and the text streams through a detokenizer. Each of these
removes overhead, and each of them adds a way to be wrong: a graph replays
what it captured, a sampler can pick from an empty set, and a stream can show
half a character.

Each ticket gives you a **symptom** and some **evidence**. Some of the
evidence is noise. Write four lines for each ticket:

1. **Root cause.** One sentence.
2. **The number that proves it.** Not "it looks like". A computation.
3. **The fix.**
4. **The guard.** A test, an assert or an alert that catches it next time.

Four rules:

- The tickets are **not** in the order of the notebooks.
- At least one ticket is **not a bug**. "Nothing is broken" is a valid answer
  only if a number proves it.
- Write your answer **before** you open the solution.
- Every ticket has a scratch cell.

Do this section after stage 14. This notebook needs no GPU.

**The on-call colleague.** In Claude Code, type `/incident 5.1` (or any
other ticket number) to work a ticket as a conversation. The colleague has
access to the system. Ask for a log, a measurement or an experiment, and it
answers with what the system shows. When you write your four lines, it tells
you which lines are weak, and it asks a question about each one. It does not
tell you the cause until you ask for the solution.

### The reference sheet

- The model is Qwen3-1.7B, on your card, unless the ticket says otherwise.
- The vocabulary has 151,936 entries. Token 0 is `!`.
- The graph buckets are the batch sizes 1, 2, 4, 8, 16 and 32. A batch is
  padded up to the next bucket.
- UTF-8 uses 1 byte for ASCII, 2 to 3 bytes for most other scripts, and 4
  bytes for an emoji.
- The replacement character `�` (U+FFFD) is what a decoder prints for bytes
  that are not a complete character.

# Ticket 1: the graph that repeats itself

**Severity:** high. **Reported by:** the team that turned on CUDA
graphs.

> With graphs on, every answer falls apart after the first token, and
> ends in `!!!!!!`, whatever the prompt. Eager mode is fine.

**Evidence**

- Five prompts give five correct first tokens. From the second token on,
  no answer follows its prompt: `' Tokyo A A A (!!!!!!!!!!'`, `' Paris A
  A A ( (!!!!!!!!!'`. Every answer ends in a run of `!`.
- The decode step with graphs:

  ```python
  # capture, once
  static_ids = torch.zeros(1, 1, dtype=torch.long, device='cuda')
  with torch.cuda.graph(graph):
      static_logits = model(static_ids, ...).logits

  # each step
  input_ids = torch.tensor([[next_token]], device='cuda')
  graph.replay()
  next_token = static_logits[0, -1].argmax().item()
  ```

- The capture ran before the warm-up. The team thinks that the order is
  the problem.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 2: the victim is always the owner of block 0

**Severity:** critical. **Reported by:** users.

> Under load, now and then one answer turns into nonsense in the middle.
> At night, with one user at a time, it never happens.

**Evidence**

- A debug dump of 9 damaged answers: each damaged request had block 0 in
  its block table.
- The batch sizes of the steps where the damage started: 3, 5, 6, 7, 9,
  12, 3, 5, 6. The logs never show damage in a step of batch 1, 2, 4 or
  8.
- The graph runner pads a batch up to the next bucket. For a padding
  row, the [slot mapping](../../GLOSSARY.md#slot-mapping) is `0`.
- The team suspects a race in the new prefix cache.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 3: graphs help at batch 1 and not at batch 64

**Severity:** low. **Reported by:** the performance team.

> CUDA graphs give 1.91x at batch 1 and only 1.11x at batch 64. The
> graphs must be broken for large batches.

**Evidence**

| batch | eager, ms for each step | graph, ms for each step |
|---|---|---|
| 1 | 24.1 | 12.6 |
| 64 | 27.9 | 25.2 |

- A profile of the eager step at batch 1: the CPU needs about 24 ms to
  launch the kernels of one step. The GPU work is about 12.6 ms.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 4: the sampler that costs more than the model

**Severity:** medium. **Reported by:** the performance team.

> The step at batch 128 takes 79 ms. The forward pass takes 38 ms. Where
> do the other 41 ms go?

**Evidence**

- The sampler time for each step: 2.6 ms at batch 8, 20.5 ms at batch 64,
  41 ms at batch 128.
- The sampler:

  ```python
  def sample(logits, requests):
      out = []
      for row, req in zip(logits, requests):
          probs = torch.softmax(row / req.temperature, -1)
          probs = apply_top_p(probs, req.top_p)
          out.append(torch.multinomial(probs, 1).item())
      return out
  ```

- A colleague says: "The vocabulary has 151,936 entries. A sampler over
  that is slow by nature."

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 5: exclamation marks when the model is sure

**Severity:** medium. **Reported by:** users.

> With `top_p=0.5`, some answers contain runs of `!!!!!!`. It happens
> more often in code.

**Evidence**

- The sampler:

  ```python
  sorted_probs, order = probs.sort(descending=True)
  cumulative = sorted_probs.cumsum(-1)
  sorted_probs[cumulative > top_p] = 0
  probs = torch.zeros_like(probs).scatter(-1, order, sorted_probs)
  token = (probs / torch.empty_like(probs).exponential_()).argmax(-1)
  ```

- The team logged 400 steps that produced `!`. In all 400 steps, the
  largest probability was above 0.5.
- Code is more predictable than prose.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 6: the seed that works only at night

**Severity:** medium. **Reported by:** a customer.

> I send `seed=42` and temperature 0.8. At night I get the same answer
> every time. During the day I get a different answer every time.

**Evidence**

- The sampler:

  ```python
  for req in batch:
      if req.seed is not None:
          torch.manual_seed(req.seed)
  tokens = torch.multinomial(probs, 1)      # the whole batch
  ```

- At night the request runs alone. During the day it shares a batch with
  20 to 60 other requests.
- An engineer says: "It is the bf16 batch effect of Part 1."

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 7: the melting face that melts

**Severity:** low. The users post screenshots. **Reported by:** the
front-end team.

> In the stream, emoji and some rare characters show as `���`. When the
> answer is complete and the page reloads, the text is correct.

**Evidence**

- The stream code:

  ```python
  for token in generated_tokens():
      yield tokenizer.decode([token])
  ```

- The emoji 🫠 appears as `���` in the stream: three replacement
  characters.
- The emoji 🙂 streams correctly.
- The front-end team changed the font of the chat last week.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 8: the words that stick together

**Severity:** medium. **Reported by:** users, after a model change.

> Since we moved from Qwen3 to Mistral-7B, the streamed text has no
> spaces: `ThecapitalofFranceisParis.` The final answer in the history is
> correct.

**Evidence**

- The stream code is the same as in Ticket 7: `tokenizer.decode([token])`
  for each token.
- Qwen3 uses a byte-level BPE tokenizer. Mistral-7B uses a SentencePiece
  tokenizer, which marks a space with `▁` at the start of a word.
- The stream has 0 spaces in the example. The final text has 5.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

# Ticket 9: the stop string that does not stop

**Severity:** medium. **Reported by:** a customer.

> We send `stop=["###"]`. Most answers stop there. Some answers show
> `###` and then continue with the next section.

**Evidence**

- The stop check:

  ```python
  text = tokenizer.decode([token])
  if any(s in text for s in request.stop):
      finish(request)
  ```

- In the answers that stop correctly, the log shows the token 14374,
  which is `###`.
- In the answers that do not stop, the log shows the token 565 (`##`)
  and then the token 2 (`#`).
- The customer thinks that the model ignores the stop instruction
  sometimes.

In [ ]:
# your computation

**Your answer**

- Root cause:
- The number that proves it:
- The fix:
- The guard (the check that catches it next time):

### Before you open the solution

Go back to each ticket and write one more line: **which piece of evidence was
noise, and why did it look relevant?**